# `00-demo` — `Dense` vs `MultiDense` on MNIST

This notebook validates the `MultiDense` custom Keras layer by comparing it against the native `Dense` layer on the MNIST handwritten digit classification task.

Both models are designed to be **mathematically equivalent**: they share the same total neuron count per layer and use identical activation functions. Any difference in results is therefore attributable to implementation divergence rather than architectural advantage — which is exactly what this demo is designed to rule out.

----

## 1. Imports

Standard TensorFlow/Keras imports alongside the custom `MultiDense` layer. `Sequential` and `Dense` are imported directly from `keras` to keep the namespace clean.

In [ ]:
import tensorflow as tf
from keras import Sequential
from keras.layers import Dense

In [ ]:
from multi_dense import MultiDense

----

## 2. Reproducibility

Setting a global random seed and enabling operation determinism ensures that results are reproducible across runs on the same machine. `set_random_seed` covers Python, NumPy, and TensorFlow's RNG simultaneously, while `enable_op_determinism` forces TensorFlow to use deterministic implementations of all ops (at a small performance cost).

> **Note:** determinism is guaranteed within a single machine and TensorFlow version, but may not transfer across different hardware or library versions.

In [ ]:
SEED = 42

In [ ]:
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

----

## 3. Model Definitions

Two models are defined for comparison:

- **Reference (`Dense`)** — a standard Keras `Sequential` model using the built-in `Dense` layer.
- **Control (`MultiDense`)** — the same architecture rebuilt using `MultiDense`, with the neurons in each hidden layer split across multiple partitions that all share the same activation.

The total neuron count and activation functions are kept identical across both models:

| Layer | `Dense` | `MultiDense` | Total neurons |
|-------|---------|--------------|---------------|
| 1     | 128 × ReLU | 50 × ReLU + 78 × ReLU | 128 |
| 2     | 64 × ReLU  | 24 × ReLU + 38 × ReLU + 2 × ReLU | 64 |
| 3     | 10 × Softmax | 10 × Softmax | 10 |

### 3.1. Reference model — native `Dense`

This is the baseline. Each layer applies a single activation function uniformly across all its neurons, which is the classical fully-connected layer behaviour.

In [ ]:
models: dict[str, Sequential] = dict()

models["Reference (Dense)"] = Sequential(
    [
        Dense(128, "relu"),
        Dense(64, "relu"),
        Dense(10, "softmax"),
    ]
)

### 3.2. Control model — `MultiDense`

Functionally identical to the reference, but each layer is expressed as a `MultiDense` with explicit partitions. Since all partitions within a layer use the same activation (`relu` for hidden layers, `softmax` for the output), this model should learn and evaluate identically to the `Dense` reference — making it a direct correctness check.

In [ ]:
models["Control (MultiDense)"] = Sequential(
    [
        MultiDense([50, 78], ["relu", "relu"]),
        MultiDense([24, 38, 2], ["relu", "relu", "relu"]),
        MultiDense([10], ["softmax"]),
    ]
)

### 3.3. Compilation

Both models are compiled with the same optimizer, loss, and metrics to ensure a fair comparison. `sparse_categorical_crossentropy` is used because the MNIST labels are integer class indices (0–9), not one-hot encoded vectors.

In [ ]:
for tag, model in models.items():
    print(f"Compiling {tag} model...")

    tf.keras.utils.set_random_seed(SEED)
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    print(f"{tag} model compiled!")
    print()

----

## 4. Dataset

MNIST is a standard benchmark dataset of 70,000 grayscale images of handwritten digits (0–9), each 28×28 pixels. The Keras API returns it pre-split into 60,000 training samples and 10,000 test samples.

### 4.1. Load

The dataset is downloaded and cached automatically on the first call.

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

### 4.2. Preprocessing

Two transformations are applied before training:

1. **Normalisation** — pixel values are rescaled from `[0, 255]` to `[0.0, 1.0]` by dividing by 255. This keeps inputs in a range that gradient-based optimisers handle well.
2. **Flattening** — each 28×28 image is reshaped into a flat vector of 784 values, since the models are fully-connected and do not exploit spatial structure.

In [ ]:
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

x_train = x_train.reshape(-1, 28**2)
x_test = x_test.reshape(-1, 28**2)

----

## 5. Training

Both models are trained sequentially with identical hyperparameters. 10% of the training data is held out as a validation split so that overfitting can be monitored during training without touching the test set.

In [ ]:
for tag, model in models.items():
    print(f"Training {tag} model...")

    tf.keras.utils.set_random_seed(SEED)
    model.fit(
        x_train,
        y_train,
        epochs=10,
        batch_size=100,
        validation_split=0.1,
    )

    print(f"{tag} model trained!")
    print()

----

## 6. Evaluation

After training, both models are evaluated on the held-out test set. Since the architectures are mathematically equivalent, their scores should be close — any large divergence would indicate a bug in `MultiDense`.

In [ ]:
models_evaluation = {
    tag: model.evaluate(x_test, y_test) for tag, model in models.items()
}

In [ ]:
for tag, (loss, accuracy) in models_evaluation.items():
    print(f"{tag} model (Loss: {loss:.4}, Accuracy: {accuracy:.2%}).")

----

## 7. Weight Swap Sanity Check

This section provides a formal correctness test for `MultiDense`. If the two implementations are truly equivalent, then swapping their trained weights must produce exactly the same evaluation scores — each model should now reproduce the other's original result.

This works because `get_weights()` returns both the kernel matrix and the bias vector, in that order, and the weight tensors are identically shaped between the two models since their total neuron counts match layer by layer.

### 7.1. Baseline — scores before the swap

Recorded first so they can be compared against the post-swap results.

In [ ]:
print("=== Before weight swap ===")

dense_loss_before, dense_acc_before = models["Reference (Dense)"].evaluate(
    x_test,
    y_test,
    verbose=0,
)
multi_loss_before, multi_acc_before = models["Control (MultiDense)"].evaluate(
    x_test,
    y_test,
    verbose=0,
)

print(f"Dense      -> Loss: {dense_loss_before:.4f}, Accuracy: {dense_acc_before:.2%}")
print(f"MultiDense -> Loss: {multi_loss_before:.4f}, Accuracy: {multi_acc_before:.2%}")

### 7.2. Perform the swap

Both sets of weights are captured before any modification, then cross-assigned. The order matters: reading both first avoids overwriting weights that have not been saved yet.

In [ ]:
dense_weights = [layer.get_weights() for layer in models["Reference (Dense)"].layers]
multi_weights = [layer.get_weights() for layer in models["Control (MultiDense)"].layers]

for layer, weights in zip(models["Control (MultiDense)"].layers, dense_weights):
    layer.set_weights(weights)

for layer, weights in zip(models["Reference (Dense)"].layers, multi_weights):
    layer.set_weights(weights)

### 7.3. Verify — scores after the swap

After the swap, `Dense` now holds `MultiDense`'s trained weights and vice versa. The assertion checks that each post-swap score matches the other model's pre-swap score exactly, confirming that `MultiDense` is a correct implementation of the same computation.

In [ ]:
import numpy as np

print("=== After weight swap ===")
dense_loss_after, dense_acc_after = models["Reference (Dense)"].evaluate(
    x_test,
    y_test,
    verbose=0,
)
multi_loss_after, multi_acc_after = models["Control (MultiDense)"].evaluate(
    x_test,
    y_test,
    verbose=0,
)
print(f"Dense      -> Loss: {dense_loss_after:.4f}, Accuracy: {dense_acc_after:.2%}")
print(f"MultiDense -> Loss: {multi_loss_after:.4f}, Accuracy: {multi_acc_after:.2%}")

dense_got_multi = np.isclose(
    dense_acc_after,
    multi_acc_before,
    atol=1e-6,
)
multi_got_dense = np.isclose(
    multi_acc_after,
    dense_acc_before,
    atol=1e-6,
)

print()
print(f"Dense(MultiDense weights) == MultiDense(original): {dense_got_multi}")
print(f"MultiDense(Dense weights) == Dense(original):      {multi_got_dense}")
print()

assert dense_got_multi, "Weight swap failed: Dense did not reproduce MultiDense's score"
assert multi_got_dense, "Weight swap failed: MultiDense did not reproduce Dense's score"
print("Sanity check passed: MultiDense is mathematically equivalent to Dense ✓")

----